# Session 31: Linear Regression: Your First Model

Data Science with Python, Level 2, Stage 5: ML Fundamentals

## Learning Objectives

By the end of this session you will be able to:

1. Explain what a machine learning model is.
2. Fit a simple linear regression.
3. Interpret slopes and the intercept.
4. Use several features in one model.
5. Split data into training and test sets.
6. Evaluate predictions with R2 and MAE.

## Where We Are

In Stage 4 you described data and tested ideas about it. **Machine learning** takes the next step: it learns a pattern from past examples and uses it to **predict** new cases. Given an apartment's size, age and location, what price will it sell for?

The simplest and still one of the most useful models is **linear regression**: it predicts a number as a weighted sum of the inputs. Stage 5 uses the 800 practice apartment sales of Session 23 as its running project.

## Setup

Run this cell first. scikit-learn, imported as `sklearn`, is the main machine learning library for Python, and it comes with Colab.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

sns.set_theme(style="whitegrid")
DATA_URL = "https://raw.githubusercontent.com/mohammadyahoo/ds-python-course/main/data/"
flats = pd.read_csv(DATA_URL + "apartments.csv")

## 1. What Is a Model?

In [ ]:
sns.scatterplot(data=flats, x="area_m2", y="price_k", s=12, alpha=0.5)
plt.title("Bigger apartments cost more")
plt.xlabel("Area (m2)")
plt.ylabel("Price (thousands)")
plt.show()

## 2. Simple Linear Regression

Every scikit-learn model works the same way: create it, `fit` it on the features and the target, then `predict`. The features `X` must be a table, even with one column, so we use double brackets.

The fitted line is `price = intercept + slope x area`. scikit-learn chooses the slope and intercept that make the squared prediction errors as small as possible; Session 32 shows how.

In [ ]:
X = flats[["area_m2"]]
y = flats["price_k"]

model = LinearRegression()
model.fit(X, y)

print("Slope:", round(model.coef_[0], 3))
print("Intercept:", round(model.intercept_, 2))

In [ ]:
new = pd.DataFrame({"area_m2": [60, 100, 150]})
print(model.predict(new).round(1))

xs = pd.DataFrame({"area_m2": np.linspace(30, 220, 50)})
plt.scatter(flats["area_m2"], flats["price_k"], s=10, alpha=0.4)
plt.plot(xs["area_m2"], model.predict(xs), color="tab:red", linewidth=2)
plt.title("The fitted line")
plt.show()

In [ ]:
flats["predicted"] = model.predict(X)
flats["residual"] = flats["price_k"] - flats["predicted"]

print(flats[["area_m2", "price_k", "predicted", "residual"]].head(4).round(1))
print("R2 on all data:", round(model.score(X, y), 3))

## 3. Multiple Regression

In [ ]:
features = ["area_m2", "rooms", "building_age", "distance_center_km", "elevator", "parking"]
X = flats[features]

model = LinearRegression().fit(X, y)
coefs = pd.Series(model.coef_, index=features).round(2)
print(coefs)
print("R2:", round(model.score(X, y), 3))

| Feature | Coefficient | Meaning, all else equal |
| --- | --- | --- |
| `area_m2` | about +0.84 | Each extra m2 adds about 840 |
| `building_age` | about -0.63 | Each year of age lowers the price by about 630 |
| `distance_center_km` | about -7.7 | Each km from the centre lowers it by about 7,700 |
| `elevator` | about +8.1 | An elevator adds about 8,100 |
| `parking` | about +8.6 | A parking space adds about 8,600 |

Prices are in thousands, so a coefficient of 0.84 means 840 per square metre.

## 4. Training and Test Data

A model that is judged on the same data it learned from can look better than it really is: it may have memorised details instead of learning the pattern. So we **split** the data: the model learns from the **training set** and is judged on the **test set**, which it never saw. `train_test_split` shuffles the rows and keeps 20 percent aside; `random_state` makes the split repeatable.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
print(len(X_train), "training rows,", len(X_test), "test rows")

model = LinearRegression().fit(X_train, y_train)
pred = model.predict(X_test)

print("R2 on test data: ", round(r2_score(y_test, pred), 3))
print("MAE on test data:", round(mean_absolute_error(y_test, pred), 2))

In [ ]:
plt.scatter(y_test, pred, s=12, alpha=0.6)
lims = [y_test.min(), y_test.max()]
plt.plot(lims, lims, color="tab:red", linestyle="--")
plt.title("Predicted against actual prices (test set)")
plt.xlabel("Actual price")
plt.ylabel("Predicted price")
plt.show()

## Common Mistakes

| Symptom | Mistake | Fix |
| --- | --- | --- |
| ValueError | Passing a Series as `X`: `flats["area_m2"]` | Use a table with double brackets: `flats[["area_m2"]]`. |
| Too optimistic | Scoring the model on its own training data | Always report scores on a test set. |
| Wrong reading | Reading a coefficient as a cause | It is an association, holding the other features fixed. |
| Leakage | Using the target, or something computed from it, as a feature | Features must be known before the price is. |

## In-Class Exercises

Work through these exercises during the session. Write your answer in the code cell under each one.

### In-Class Exercise 1: One Feature at a Time

Fit a simple regression of price on each of `rooms`, `building_age` and `distance_center_km` separately. Which single feature explains the most (highest R2)?

In [ ]:
# Write your code below

### In-Class Exercise 2: Predict Your Flat

Using the multiple regression trained on the training set, predict the price of a 90 m2 flat with 3 rooms, 10 years old, 4 km from the centre, with an elevator and no parking.

In [ ]:
# Write your code below

### In-Class Exercise 3: Biggest Errors

Find the five test flats with the largest absolute prediction errors. What do they have in common?

In [ ]:
# Write your code below

### In-Class Exercise 4: Different Split

Repeat the train-test split with `random_state` 1, 2 and 3 and print the test R2 each time. How stable is the score?

In [ ]:
# Write your code below

## Summary

- A model learns from examples to predict a target from features.
- Linear regression predicts with a weighted sum of the features.
- Every scikit-learn model: create, `fit`, `predict`.
- Coefficients show each feature's effect, others held fixed.
- Train on one part of the data, test on the rest.
- R2 measures explained variation; MAE the typical error.

## What's Next

Today `fit` found the best line instantly. Next session you will open the box: define what best means with a loss function, and write gradient descent, the method that trains almost every modern model.